# OlmoEarth Embeddings Pipeline

Compute [OlmoEarth](https://github.com/allenai/rslearn) foundation-model embeddings over a
user-defined area of interest (AOI) using the Allen AI `rslearn` library.

## Workflow

| Step | What happens |
|------|--------------|
| USER INPUTS | Edit one cell — shapefile, dates, satellites |
| 1 | Load shapefile, reproject to EPSG:4326, plot AOI |
| 2 | Compute `period_duration` for 12 equal mosaics |
| 3 | Build and write `config.json` (dataset config) |
| 4 | Add `embeddings` output layer to `config.json` |
| 5 | Create rslearn windows (`add_windows`) |
| 6 | Materialise satellite imagery (`prepare` + `materialize`) |
| 7 | Build and write `model.yaml` (inference config) |
| 8 | Run `rslearn model predict` to write embedding GeoTIFFs |
| 9 | Load and visualise the output embeddings |

## AWS credentials (Landsat 8 only)

If `'landsat8'` is in `SATELLITES`, set these **before** starting the notebook:

```bash
export AWS_ACCESS_KEY_ID=your_key
export AWS_SECRET_ACCESS_KEY=your_secret
```

The `usgs-landsat` S3 bucket is requester-pays. Sentinel-2 and Sentinel-1 use the
Microsoft Planetary Computer API and require no credentials.

## Example USER INPUTS

```python
SHAPEFILE_PATH = '/data/my_aoi.shp'
START_DATE     = '2024-01-01'
END_DATE       = '2024-07-01'
SATELLITES     = ['s2', 's1']          # any subset of ['s2', 's1', 'landsat8']
DATASET_PATH   = './olmoearth_dataset'
```

## Note on timestep density

This notebook divides the date range into **12 equal mosaics** (≈ 15 days each for a
6-month window). OlmoEarth was pre-trained on 30-day mosaics, so 12 timesteps over
6 months is slightly out-of-distribution. Compute time also scales with token sequence
length — more timesteps mean longer sequences per patch.

In [ ]:
# Install all required dependencies (safe to re-run)
%pip install 'rslearn[extra]' geopandas rasterio matplotlib pyyaml shapely --quiet

In [ ]:
import glob
import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import yaml


def stream_cmd(cmd):
    """Run a shell command, streaming stdout+stderr line-by-line into the notebook."""
    proc = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

---

## USER INPUTS

Edit **only** the cell below. All downstream cells derive from these five variables.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# USER INPUTS — Edit only this cell
# ══════════════════════════════════════════════════════════════════════

SHAPEFILE_PATH = '/path/to/your/aoi.shp'   # polygon shapefile for area of interest

START_DATE = '2024-01-01'                   # start of time range (ISO date, inclusive)
END_DATE   = '2024-07-01'                   # end of time range   (ISO date, exclusive)

SATELLITES = ['s2', 's1']                  # any subset of ['s2', 's1', 'landsat8']
#   's2'       → Sentinel-2 L2A     (Planetary Computer, no credentials needed)
#   's1'       → Sentinel-1 IW SAR  (Planetary Computer, no credentials needed)
#   'landsat8' → Landsat 8/9 OLI-TIRS (AWS usgs-landsat, requires AWS credentials)

DATASET_PATH = './olmoearth_dataset'        # directory where the dataset will be created

# ══════════════════════════════════════════════════════════════════════

---

## Step 1 — Load AOI and visualise

Read the shapefile, reproject to EPSG:4326, extract the bounding box, and plot the
polygon so you can visually confirm the correct area was loaded before running any
expensive commands.

In [ ]:
aoi = gpd.read_file(SHAPEFILE_PATH)
aoi_4326 = aoi.to_crs('EPSG:4326')

minx, miny, maxx, maxy = aoi_4326.total_bounds

print(f'Features loaded : {len(aoi_4326)}')
print(f'Original CRS    : {aoi.crs}')
print(f'Bounding box (EPSG:4326):')
print(f'  lon : [{minx:.6f},  {maxx:.6f}]')
print(f'  lat : [{miny:.6f},  {maxy:.6f}]')

fig, ax = plt.subplots(figsize=(8, 6))
aoi_4326.plot(ax=ax, color='steelblue', alpha=0.35, edgecolor='navy', linewidth=1.5)
ax.set_title('Area of Interest', fontsize=14)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

In [ ]:
# Check AWS credentials early — fail fast if Landsat is selected but credentials absent.
# Landsat 8 uses the usgs-landsat S3 requester-pays bucket.
if 'landsat8' in SATELLITES:
    missing_vars = [v for v in ('AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY')
                    if not os.environ.get(v)]
    if missing_vars:
        raise EnvironmentError(
            f'Landsat 8 selected but AWS credentials not found in environment: {missing_vars}. '
            'Before starting this notebook, set AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY '
            'in your shell. Do NOT hardcode credentials in this notebook.'
        )
    print('AWS credentials present — Landsat 8 access confirmed.')
else:
    print('Landsat 8 not selected — AWS credential check skipped.')

---

## Step 2 — Compute `period_duration`

OlmoEarth was pre-trained on per-period mosaics with `period_duration = "30d"` (one
mosaic per calendar month). We want **12 equal timesteps** across the user-supplied date
range, so:

```
period_duration = floor((END_DATE − START_DATE) / 12) days
```

For a 6-month window this gives ≈ 15 days per mosaic.

### `max_matches` vs number of timesteps — two independent concepts

- **`period_duration`** controls *how many mosaics* (timesteps) are produced — one per
  period within the date range. With 12 periods we get 12 rslearn layer instances:
  `sentinel2_l2a`, `sentinel2_l2a.1`, … `sentinel2_l2a.11`.
- **`max_matches`** is the **per-mosaic cap on source scenes** that contribute to *each
  individual composite*. We set it to 12, meaning up to 12 raw satellite acquisitions
  may be mosaicked together per period (useful for cloud avoidance). Changing
  `max_matches` does not change the number of timesteps.

In [ ]:
start_dt = datetime.fromisoformat(START_DATE)
end_dt   = datetime.fromisoformat(END_DATE)

total_days    = (end_dt - start_dt).days
NUM_TIMESTEPS = 12
period_days   = total_days // NUM_TIMESTEPS   # whole days, round down

period_duration = f'{period_days}d'

print(f'Date range       : {START_DATE}  →  {END_DATE}  ({total_days} days)')
print(f'Target timesteps : {NUM_TIMESTEPS}')
print(f'period_duration  : {period_duration}  ({period_days} days per mosaic)')
print(f'max_matches      : 12  (per-mosaic source-scene cap, independent of timestep count)')

---

## Step 3 — Build `config.json`

The rslearn dataset configuration specifies which satellite layers to retrieve and how
to mosaic them. We build it programmatically from the USER INPUTS and write it to
`{DATASET_PATH}/config.json`.

Key fields per layer:

| Field | Value | Meaning |
|-------|-------|---------|
| `space_mode` | `PER_PERIOD_MOSAIC` | Create one composite per period |
| `period_duration` | computed above | Length of each mosaic window |
| `max_matches` | `12` | Max source scenes per mosaic (independent of timestep count) |
| `ingest` | `false` | Stream tiles on demand; do not copy to a local tile store |

Only layers for the selected satellites are included.

In [ ]:
Path(DATASET_PATH).mkdir(parents=True, exist_ok=True)

layer_configs = {}

if 's2' in SATELLITES:
    layer_configs['sentinel2_l2a'] = {
        'band_sets': [{
            'bands': ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12'],
            'dtype': 'uint16',
        }],
        'data_source': {
            'class_path': 'rslearn.data_sources.planetary_computer.Sentinel2',
            'init_args': {
                'cache_dir': 'cache/planetary_computer',
                'harmonize': True,
                'sort_by': 'eo:cloud_cover',
            },
            'ingest': False,
            'query_config': {
                'max_matches': 12,
                'period_duration': period_duration,
                'space_mode': 'PER_PERIOD_MOSAIC',
            },
        },
        'type': 'raster',
    }

if 's1' in SATELLITES:
    layer_configs['sentinel1'] = {
        'band_sets': [{
            'bands': ['vv', 'vh'],
            'dtype': 'float32',
            'nodata_vals': [-32768, -32768],
        }],
        'data_source': {
            'class_path': 'rslearn.data_sources.planetary_computer.Sentinel1',
            'init_args': {
                'cache_dir': 'cache/planetary_computer',
                'query': {
                    'sar:instrument_mode': {'eq': 'IW'},
                    'sar:polarizations': {'eq': ['VV', 'VH']},
                },
            },
            'ingest': False,
            'query_config': {
                'max_matches': 12,
                'period_duration': period_duration,
                'space_mode': 'PER_PERIOD_MOSAIC',
            },
        },
        'type': 'raster',
    }

if 'landsat8' in SATELLITES:
    layer_configs['landsat'] = {
        'band_sets': [{
            'bands': ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11'],
            'dtype': 'uint16',
        }],
        'data_source': {
            'class_path': 'rslearn.data_sources.aws_landsat.LandsatOliTirs',
            'init_args': {
                'metadata_cache_dir': 'cache/landsat',
                'sort_by': 'cloud_cover',
            },
            'ingest': False,
            'query_config': {
                'max_matches': 12,
                'period_duration': period_duration,
                'space_mode': 'PER_PERIOD_MOSAIC',
            },
        },
        'type': 'raster',
    }

dataset_config = {'layers': layer_configs}
config_path = Path(DATASET_PATH) / 'config.json'

with open(config_path, 'w') as fh:
    json.dump(dataset_config, fh, indent=2)

print(f'config.json written to: {config_path}')
print()
print(json.dumps(dataset_config, indent=2))

---

## Step 4 — Add `embeddings` output layer to `config.json`

The `rslearn model predict` step writes embedding GeoTIFFs into a layer named
`embeddings`. We register it in `config.json` now so rslearn knows the output schema:
768 float32 bands, matching `OLMOEARTH_V1_BASE`.

| Model variant | `num_bands` |
|--------------|-------------|
| OLMOEARTH_V1_NANO  | 128  |
| OLMOEARTH_V1_TINY  | 192  |
| OLMOEARTH_V1_BASE  | 768  |
| OLMOEARTH_V1_LARGE | 1024 |

In [ ]:
dataset_config['layers']['embeddings'] = {
    'band_sets': [{'dtype': 'float32', 'num_bands': 768}],
    'type': 'raster',
}

with open(config_path, 'w') as fh:
    json.dump(dataset_config, fh, indent=2)

print('"embeddings" layer added and config.json updated.')
print()
print(json.dumps({'embeddings': dataset_config['layers']['embeddings']}, indent=2))

---

## Step 5 — Create rslearn windows (`add_windows`)

A **window** in rslearn is a spatiotemporal box — a geographic region paired with a
time range — that defines one inference unit. We create window(s) for the AOI bounding
box.

- **`--utm --resolution 10`**: 10 m/pixel in UTM projection, matching OlmoEarth
  pre-training.
- **`--src_crs EPSG:4326`**: the `--box` coordinates are in lon/lat.
- **`--box lon1,lat1,lon2,lat2`**: bounding box derived from the shapefile.
- **`--grid_size 1024`**: automatically added if the AOI exceeds 10 km × 10 km, which
  splits the area into multiple 1024 × 1024 pixel tiles to keep memory manageable.

In [ ]:
# Measure AOI in UTM to decide whether --grid_size is needed
aoi_utm = aoi_4326.to_crs(aoi_4326.estimate_utm_crs())
bx0, by0, bx1, by1 = aoi_utm.total_bounds
width_km  = (bx1 - bx0) / 1000
height_km = (by1 - by0) / 1000
use_grid  = (width_km > 10) or (height_km > 10)

print(f'AOI extent : {width_km:.2f} km  x  {height_km:.2f} km')
print(f'--grid_size: {"1024  (AOI exceeds 10 km x 10 km — multiple tiles will be created)" if use_grid else "not added  (AOI fits in a single window)"}')

grid_arg = '--grid_size 1024' if use_grid else ''

cmd = (
    f'rslearn dataset add_windows'
    f' --root {DATASET_PATH}'
    f' --group default --name default'
    f' --utm --resolution 10 --src_crs EPSG:4326'
    f' --box={minx},{miny},{maxx},{maxy}'
    f' --start {START_DATE}T00:00:00+00:00'
    f' --end {END_DATE}T00:00:00+00:00'
    + (f' {grid_arg}' if grid_arg else '')
)

print()
print('Running:')
print(f'  {cmd}')
print()
stream_cmd(cmd)
print()
print('Windows added successfully.')

---

## Step 6 — Materialise satellite imagery

Two rslearn commands download and align the satellite images for the windows:

1. **`prepare`** — queries each data source to find which raw scenes (Sentinel-2 tiles,
   Sentinel-1 granules, Landsat scenes) intersect each window and each time period.
2. **`materialize`** — downloads those scenes, reprojects them to the window's UTM CRS,
   crops to the window extent, and writes GeoTIFFs.

After this step, time-series GeoTIFFs appear under:

```
{DATASET_PATH}/windows/default/<window>/layers/<layer_name>/
```

With 12 timesteps you will see `sentinel2_l2a/`, `sentinel2_l2a.1/`, …,
`sentinel2_l2a.11/` (and equivalents for other selected satellites).

Runtime depends on AOI size, date range, and network speed — expect minutes to hours.

In [ ]:
LAYER_NAME_MAP = {'s2': 'sentinel2_l2a', 's1': 'sentinel1', 'landsat8': 'landsat'}
enabled_layers = ','.join(LAYER_NAME_MAP[s] for s in SATELLITES if s in LAYER_NAME_MAP)

print(f'Enabled layers: {enabled_layers}')
print()

prepare_cmd = (
    f'rslearn dataset prepare'
    f' --root {DATASET_PATH}'
    f' --workers 32'
    f' --enabled-layers {enabled_layers}'
    f' --retry-max-attempts 5'
    f' --retry-backoff-seconds 5'
)

materialize_cmd = (
    f'rslearn dataset materialize'
    f' --root {DATASET_PATH}'
    f' --workers 32'
    f' --no-use-initial-job'
    f' --enabled-layers {enabled_layers}'
    f' --retry-max-attempts 5'
    f' --retry-backoff-seconds 5'
)

print('PREPARE ─────────────────────────────────────────────────────')
print(f'  {prepare_cmd}')
print()
stream_cmd(prepare_cmd)

print()
print('MATERIALIZE ─────────────────────────────────────────────────')
print(f'  {materialize_cmd}')
print()
stream_cmd(materialize_cmd)

print()
print('Materialisation complete.')

---

## Step 7 — Generate `model.yaml`

The model configuration drives inference. Key elements:

- **`OlmoEarth` encoder** — `model_id: OLMOEARTH_V1_BASE`, `patch_size: 4`.
- **`EmbeddingHead` + `EmbeddingTask`** — a pass-through task that routes the encoder
  feature map to the output writer without any classification head.
- **Inputs** — one entry per satellite modality. Each input enumerates **all 12
  timestep layer names** (e.g. `sentinel2_l2a`, `sentinel2_l2a.1`, …
  `sentinel2_l2a.11`). Bands are listed in the exact order OlmoEarth expects, which
  differs from the dataset storage order for Sentinel-2.
- **`OlmoEarthNormalize`** — normalises each band to the statistics used during
  pre-training; applied per modality with the same band ordering.
- **Sliding-window inference** — `crop_size 64`, `overlap_pixels 32` at input
  resolution (64 × 64 px crops with 32 px overlap). The `RasterMerger` trims 8 pixels
  of border at output resolution (= 32 input pixels ÷ patch_size 4) and uses
  `downsample_factor 4` since the encoder produces one embedding per 4 × 4 input patch.

### Band order expected by OlmoEarth

| Modality | Band order |
|----------|------------|
| Sentinel-2 L2A | `B02 B03 B04 B08 B05 B06 B07 B8A B11 B12 B01 B09` (RGB+NIR first, then red-edge, SWIR, coastal/water-vapour) |
| Sentinel-1 | `vv vh` |
| Landsat 8/9 | `B1 B2 B3 B4 B5 B6 B7 B8 B9 B10 B11` |

In [ ]:
def make_timestep_layers(base_name, n=NUM_TIMESTEPS):
    """Return ['base_name', 'base_name.1', ..., 'base_name.{n-1}']."""
    return [base_name] + [f'{base_name}.{i}' for i in range(1, n)]


# Exact band orders expected by OlmoEarth (from the official tutorial)
S2_BANDS      = ['B02', 'B03', 'B04', 'B08', 'B05', 'B06', 'B07', 'B8A', 'B11', 'B12', 'B01', 'B09']
S1_BANDS      = ['vv', 'vh']
LANDSAT_BANDS = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11']

inputs = {}
band_names_for_norm = {}

if 's2' in SATELLITES:
    inputs['sentinel2_l2a'] = {
        'data_type': 'raster',
        'layers': make_timestep_layers('sentinel2_l2a'),
        'bands': S2_BANDS,
        'passthrough': True,
        'dtype': 'FLOAT32',
        'load_all_layers': True,
    }
    band_names_for_norm['sentinel2_l2a'] = S2_BANDS

if 's1' in SATELLITES:
    inputs['sentinel1'] = {
        'data_type': 'raster',
        'layers': make_timestep_layers('sentinel1'),
        'bands': S1_BANDS,
        'passthrough': True,
        'dtype': 'FLOAT32',
        'load_all_layers': True,
    }
    band_names_for_norm['sentinel1'] = S1_BANDS

if 'landsat8' in SATELLITES:
    inputs['landsat'] = {
        'data_type': 'raster',
        'layers': make_timestep_layers('landsat'),
        'bands': LANDSAT_BANDS,
        'passthrough': True,
        'dtype': 'FLOAT32',
        'load_all_layers': True,
    }
    band_names_for_norm['landsat'] = LANDSAT_BANDS

print('Timestep layer lists (12 per modality):')
for key, cfg in inputs.items():
    print(f'  {key}: {cfg["layers"]}')
print()

model_config = {
    'model': {
        'class_path': 'rslearn.train.lightning_module.RslearnLightningModule',
        'init_args': {
            'model': {
                'class_path': 'rslearn.models.singletask.SingleTaskModel',
                'init_args': {
                    'encoder': [{
                        'class_path': 'rslearn.models.olmoearth_pretrain.model.OlmoEarth',
                        'init_args': {
                            'model_id': 'OLMOEARTH_V1_BASE',
                            'patch_size': 4,
                        },
                    }],
                    'decoder': [{
                        'class_path': 'rslearn.train.tasks.embedding.EmbeddingHead',
                    }],
                },
            },
            'optimizer': {
                'class_path': 'rslearn.train.optimizer.AdamW',
            },
        },
    },
    'data': {
        'class_path': 'rslearn.train.data_module.RslearnDataModule',
        'init_args': {
            'path': str(Path(DATASET_PATH).resolve()),
            'inputs': inputs,
            'task': {
                'class_path': 'rslearn.train.tasks.embedding.EmbeddingTask',
            },
            'batch_size': 8,
            'num_workers': 32,
            'predict_config': {
                'transforms': [{
                    'class_path': 'rslearn.models.olmoearth_pretrain.norm.OlmoEarthNormalize',
                    'init_args': {
                        'band_names': band_names_for_norm,
                    },
                }],
                'load_all_crops': True,
                'crop_size': 64,
                'overlap_pixels': 32,
            },
        },
    },
    'trainer': {
        'callbacks': [{
            'class_path': 'rslearn.train.prediction_writer.RslearnWriter',
            'init_args': {
                'output_layer': 'embeddings',
                'merger': {
                    'class_path': 'rslearn.train.prediction_writer.RasterMerger',
                    'init_args': {
                        'overlap_pixels': 8,
                        'downsample_factor': 4,
                    },
                },
            },
        }],
    },
}

model_yaml_path = 'model.yaml'
with open(model_yaml_path, 'w') as fh:
    yaml.dump(model_config, fh, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f'model.yaml written to: {model_yaml_path}')
print()
with open(model_yaml_path) as fh:
    print(fh.read())

---

## Step 8 — Compute embeddings (`rslearn model predict`)

This runs the OlmoEarth encoder over all windows using sliding-window inference and
writes 768-band embedding GeoTIFFs to the `embeddings` layer.

Output files will appear at:
```
{DATASET_PATH}/windows/default/<window>/layers/embeddings/*/geotiff.tif
```

A GPU is required for practical inference speed. Runtime scales with the number of
windows (AOI size) and the number of timesteps.

In [ ]:
predict_cmd = 'rslearn model predict --config model.yaml'
print('Running:')
print(f'  {predict_cmd}')
print()
stream_cmd(predict_cmd)
print()
print('Embedding prediction complete.')

---

## Step 9 — Inspect output embeddings

Load one of the output embedding GeoTIFFs, print its shape and geospatial metadata,
and display a quick false-colour visualisation of the first three embedding dimensions
to confirm the pipeline completed successfully.

In [ ]:
embed_pattern = str(
    Path(DATASET_PATH) / 'windows' / 'default' / '*' / 'layers' / 'embeddings' / '*' / 'geotiff.tif'
)
embed_files = sorted(glob.glob(embed_pattern))

if not embed_files:
    print('No embedding GeoTIFFs found. Confirm that Step 8 completed without errors.')
    print(f'Expected pattern: {embed_pattern}')
else:
    tif_path = embed_files[0]
    print(f'Found {len(embed_files)} embedding file(s).')
    print(f'Loading first file: {tif_path}')
    print()

    with rasterio.open(tif_path) as src:
        n_bands = src.count
        height  = src.height
        width   = src.width
        crs     = src.crs
        bounds  = src.bounds
        rgb_raw = src.read([1, 2, 3]).astype(np.float32)

    print(f'Shape  : {n_bands} bands  x  {height} rows  x  {width} cols')
    print(f'CRS    : {crs}')
    print(f'Bounds : {bounds}')
    print()

    # Percentile-stretch each of the three channels independently for display
    rgb = np.zeros_like(rgb_raw)
    for i in range(3):
        ch = rgb_raw[i]
        lo, hi = np.nanpercentile(ch, 2), np.nanpercentile(ch, 98)
        rgb[i] = np.clip((ch - lo) / (hi - lo + 1e-9), 0.0, 1.0)

    rgb_img = np.moveaxis(rgb, 0, -1)   # (H, W, 3) for imshow

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(rgb_img)
    axes[0].set_title('False-colour: embedding channels 1-3')
    axes[0].axis('off')

    axes[1].imshow(rgb[0], cmap='viridis')
    axes[1].set_title('Embedding channel 1 (viridis)')
    axes[1].axis('off')

    window_name = Path(tif_path).parts[-5]
    plt.suptitle(f'OlmoEarth Embeddings  —  window: {window_name}', fontsize=13)
    plt.tight_layout()
    plt.show()

    if len(embed_files) > 1:
        print(f'{len(embed_files)} windows found in total; showing the first above.')
        print('All embedding files:')
        for p in embed_files:
            print(f'  {p}')